In [5]:
import pandas as pd

# Lista dei tickers da tenere (filtro opzionale)
tickers = [
    'ATOM','REQ','CRV','MAVIA','SAGA','NEAR','MORPHO','MANTA','MOVE','XAI',
    'ETC','DOGE','SOPH','CELO','MAV','POPCAT','SCR','COMP','GMT','SOL','IMX',
    'JUP','RUNE','LAUNCHCOIN','UMA','TRB','USTC','AIXBT','IOTA','VIRTUAL',
    'ALGO','GMX','ANIME','BCH','BIO','BSV','NXPC','MOODENG','TNSR','HBAR',
    'SNX','ZEREBRO','HYPER','SAND','BERA','PURR','GAS','LDO','ONDO','DYDX',
    'FTT','TON','EIGEN','LTC','BLAST','AI16Z','OMNI','AAVE','OGN','SUI',
    'MEME','FXS','NEIROETH','NIL','CFX','ME','XRP','TIA','BNB','NOT','IP',
    'OM','TAO','OP','CAKE','AVAX','kPEPE','GALA','MNT','BOME','SUPER','SEI',
    'VINE','KAS','BABY','STX','S','FARTCOIN','STG','RENDER','ENA','LINK',
    'ARB','ARK','BIGTIME','BTC','ETH','RSR','kDOGS','BRETT','BANANA','XLM',
    'INJ','ENS','AR','DOT','SPX','ETHFI','PAXG','kLUNC','GOAT','kSHIB','FIL',
    'MEW','STRK','TRX','ZK','KAITO','PENGU','kBONK','VVV','ORDI','INIT','APT',
    'REZ','LAYER','ZEN','SUSHI','kFLOKI','ADA','kNEIRO','PEOPLE','ZORA',
    'PENDLE','APE','HYPE','FET','CHILLGUY','MELANIA','GRIFFAIN','PNUT','DOOD',
    'WIF','ACE','ZETA','TRUMP','NEO','JTO','YGG','ZRO','PROMPT','WLD','W',
    'MERL','BLUR','UNI','DYM','MINA','MKR','POLYX','POL','IO','TURBO','PYTH',
    'USUAL','GRASS','ALT','HMSTR','WCT','SYRUP','RESOLV','PROVE','YZY','WLFI',
    'TST','PUMP','LINEA','SKY','ASTER','0G','STBL','AVNT','XPL','ZEC','ICP'
]

# 1. Carico i dati
funding = pd.read_csv("../data/hourly_funding.csv")
oracle  = pd.read_csv("../data/oracle_price.csv")

# 2. Parsing del timestamp (formato ISO8601 misto + timezone)
funding["time"] = pd.to_datetime(funding["time"], format="ISO8601", utc=True)
oracle["time"]  = pd.to_datetime(oracle["time"],  format="ISO8601", utc=True)

# Se vuoi rimuovere l'informazione di timezone:
# funding["time"] = funding["time"].dt.tz_convert("UTC").dt.tz_localize(None)
# oracle["time"]  = oracle["time"].dt.tz_convert("UTC").dt.tz_localize(None)

# 3. Filtro opzionale sui tickers
funding = funding[funding["perp"].isin(tickers)]
oracle  = oracle[oracle["perp"].isin(tickers)]

# 4. (Opzionale) rimuovo eventuali duplicati su (perp, time)
funding = funding.drop_duplicates(subset=["perp", "time"])
oracle  = oracle.drop_duplicates(subset=["perp", "time"])

# 5. Merge su perp + time
merged = pd.merge(
    funding,
    oracle,
    on=["perp", "time"],
    how="inner",          # solo le coppie (perp,time) presenti in entrambi
    validate="one_to_one" # togli questa riga se hai ancora duplicati
)

# 6. Calcolo del prodotto ora-per-ora
merged["funding_px_product"] = merged["fundingRate"] * merged["oraclePx"]

# 7. Ordino e salvo
merged = merged.sort_values(["perp", "time"])
merged.to_csv("payment_factor.csv", index=False)

# Se vuoi anche una versione "wide":
# wide = merged.pivot(index="time", columns="perp", values="funding_px_product")
# wide.to_csv("payment_factor_wide.csv")
